# ViT finetuning

## Installing Dependencies

In [1]:
!pip install pytorch_metric_learning torchmetrics timm albumentations

## Imports

In [2]:
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pytorch_metric_learning import miners, losses
from torchmetrics import Accuracy
from tqdm import tqdm
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import timm
from torch.cuda.amp import autocast, GradScaler

/usr/local/lib/python3.10/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.5 (you have 1.4.20). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


## Configurations

In [3]:
class Config:
    BASE_DIR = "/kaggle/input/mot20fawry/face_identification/face_identification"
    TRAIN_CSV = os.path.join(BASE_DIR, "trainset.csv")
    TEST_CSV = os.path.join(BASE_DIR, "eval_set.csv")
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    IMAGE_DIR = BASE_DIR
    
    BATCH_SIZE = 32
    NUM_EPOCHS = 60
    LEARNING_RATE = 3e-5
    EMBEDDING_DIM = 1024
    NUM_FOLDS = 5
    MARGIN = 0.5  
    SCALE = 64.0  
    TRIPLET_MARGIN = 0.5
    IMG_SIZE = 224
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Running on {Config.DEVICE}")

Running on cuda


## Dataset Helper Class

In [4]:
class AdvancedRetailFaceDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.train = train
        if self.train:
            self.transform = A.Compose([
                A.Resize(224, 224),
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.Rotate(limit=20, p=0.5),
                A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=0.5),
                A.GaussNoise(p=0.3),
                A.CoarseDropout(max_holes=2, max_height=16, max_width=16, min_holes=1, min_height=8, min_width=8, p=0.5),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ])
        else:
            self.transform = A.Compose([
                A.Resize(224, 224),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['full_path']
        label = self.df.iloc[idx]['label']
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            return None, None
        image = np.array(image)
        augmented = self.transform(image=image)
        return augmented['image'], label

## Model Definition + CurricularFace

In [5]:
class AdvancedFaceReIDModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.embedding_layer = nn.Sequential(
            nn.Linear(768, 2048),
            nn.BatchNorm1d(2048),
            nn.PReLU(),
            nn.Dropout(0.6),
            nn.Linear(2048, Config.EMBEDDING_DIM)
        )
        self.curricularface = CurricularFace(in_features=Config.EMBEDDING_DIM, out_features=num_classes, scale=Config.SCALE, margin=Config.MARGIN)
        self.classifier = nn.Linear(Config.EMBEDDING_DIM, num_classes)

    def forward(self, x, labels=None):
        features = self.backbone(x)
        embeddings = self.embedding_layer(features)
        classifier_logits = self.classifier(embeddings)
        curricular_logits = self.curricularface(embeddings, labels) if labels is not None else None
        return embeddings, curricular_logits, classifier_logits


class CurricularFace(nn.Module):
    def __init__(self, in_features, out_features, scale=64.0, margin=0.5):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.scale = scale
        self.margin = margin

    def forward(self, embeddings, labels):
        embeddings = nn.functional.normalize(embeddings, p=2, dim=1)
        weight = nn.functional.normalize(self.weight, p=2, dim=1)
        cos_theta = torch.matmul(embeddings, weight.T)
        
        margin_tensor = torch.tensor(self.margin, dtype=cos_theta.dtype, device=Config.DEVICE)
        cos_m = torch.cos(margin_tensor)
        sin_m = torch.sin(margin_tensor)
        threshold = torch.cos(torch.tensor(np.pi, dtype=cos_theta.dtype, device=Config.DEVICE) - margin_tensor)
        mm = sin_m * margin_tensor
        
        sin_theta = torch.sqrt(1.0 - torch.pow(cos_theta, 2) + 1e-6)
        cos_theta_m = cos_theta * cos_m - sin_theta * sin_m
        mask = cos_theta > threshold
        
        # Fixed line using torch.where
        cos_theta_m = torch.where(mask, cos_theta - mm, cos_theta_m)
        
        one_hot = nn.functional.one_hot(labels, num_classes=self.weight.size(0)).float().to(Config.DEVICE)
        output = self.scale * (one_hot * cos_theta_m + (1 - one_hot) * cos_theta)
        return output

## Finetuning

In [7]:
def train_fold(fold, train_idx, val_idx, df, num_classes):
    model = AdvancedFaceReIDModel(num_classes).to(Config.DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.NUM_EPOCHS)
    
    print(f"Training fold {fold+1} from scratch with CurricularFace")
    
    train_labels = df.iloc[train_idx]['label'].values
    class_counts = np.bincount(train_labels, minlength=num_classes)
    class_weights = 1. / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum()
    ce_criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(Config.DEVICE), label_smoothing=0.1)
    triplet_criterion = losses.TripletMarginLoss(margin=Config.TRIPLET_MARGIN)
    contrastive_criterion = losses.ContrastiveLoss(pos_margin=0.1, neg_margin=1.0)
    
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    train_dataset = AdvancedRetailFaceDataset(df.iloc[train_idx], train=True)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=2)
    val_dataset = AdvancedRetailFaceDataset(df.iloc[val_idx], train=False)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
    
    train_acc_metric = Accuracy(task="multiclass", num_classes=num_classes).to(Config.DEVICE)
    val_acc_metric = Accuracy(task="multiclass", num_classes=num_classes).to(Config.DEVICE)
    
    miner = miners.MultiSimilarityMiner()
    best_acc = 0.0
    scaler = torch.amp.GradScaler('cuda')
    
    with tqdm(total=Config.NUM_EPOCHS, desc=f"Fold {fold+1}", position=0, leave=True) as epoch_pbar:
        for epoch in range(Config.NUM_EPOCHS):
            model.train()
            total_loss = 0
            with tqdm(train_loader, desc=f"Training Epoch {epoch+1}/{Config.NUM_EPOCHS}", position=0, leave=False) as train_pbar:
                for images, labels in train_pbar:
                    if images is None or labels is None:
                        continue
                    images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
                    optimizer.zero_grad()
                    with torch.amp.autocast('cuda'):
                        embeddings, curricular_logits, classifier_logits = model(images, labels)
                        ce_loss = ce_criterion(classifier_logits, labels)
                        curricular_loss = ce_criterion(curricular_logits, labels)
                        hard_pairs = miner(embeddings, labels)
                        triplet_loss = triplet_criterion(embeddings, labels, hard_pairs)
                        contrastive_loss = contrastive_criterion(embeddings, labels)
                        loss = ce_loss + curricular_loss + triplet_loss + contrastive_loss
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                    total_loss += loss.item()
                    preds = torch.argmax(classifier_logits, dim=1)
                    train_acc_metric.update(preds, labels)
                    train_pbar.set_postfix({'loss': f"{total_loss / (train_pbar.n + 1):.4f}"})
            
            train_acc = train_acc_metric.compute().item()
            train_acc_metric.reset()
            scheduler.step()
            
            model.eval()
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    for images, labels in val_loader:
                        if images is None or labels is None:
                            continue
                        images, labels = images.to(Config.DEVICE), labels.to(Config.DEVICE)
                        _, _, classifier_logits = model(images)
                        preds = torch.argmax(classifier_logits, dim=1)
                        val_acc_metric.update(preds, labels)
            val_acc = val_acc_metric.compute().item()
            val_acc_metric.reset()
            
            epoch_pbar.set_postfix({
                'Train Acc': f"{train_acc * 100:.2f}%",  # Multiply by 100, 2 decimal places, add % symbol
                'Loss': f"{total_loss / len(train_loader):.4f}",
                'Val Acc': f"{val_acc * 100:.2f}%"       # Multiply by 100, 2 decimal places, add % symbol
            })
            epoch_pbar.update(1)
            
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save({
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'epoch': epoch
                }, f"{Config.CHECKPOINT_DIR}/fold_{fold+1}_best.pth")
                print(f"Model Saved With Accuracy{best_acc* 100:.2f}%")
    
    return best_acc

## Main

In [ ]:
def main():
    try:
        train_df = pd.read_csv(Config.TRAIN_CSV, sep=',', engine='python', header=0, names=['image_path', 'gt'], on_bad_lines='skip')
        train_df = train_df.dropna()
        train_df['image_path'] = train_df['image_path'].str.strip()
        train_df['gt'] = train_df['gt'].str.strip()
        train_df['full_path'] = train_df['image_path'].apply(lambda x: os.path.join(Config.BASE_DIR, x))
        le = LabelEncoder()
        train_df['label'] = le.fit_transform(train_df['gt'])
        valid_paths = train_df['full_path'].apply(os.path.exists)
        if not valid_paths.all():
            invalid_paths = train_df[~valid_paths]['full_path'].tolist()
            print(f"Warning: {len(invalid_paths)} invalid training paths found: {invalid_paths[:5]}")
            train_df = train_df[valid_paths]
        print(f"Total valid training samples: {len(train_df)}")
    except Exception as e:
        print(f"Data loading error: {e}")
        raise

    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    skf = StratifiedKFold(n_splits=Config.NUM_FOLDS, shuffle=True, random_state=42)
    fold_accuracies = []
    for fold, (train_idx, val_idx) in tqdm(enumerate(skf.split(train_df, train_df['gt'])), total=Config.NUM_FOLDS, desc="Training"):
        print(f"\nTraining Fold {fold+1}")
        acc = train_fold(fold, train_idx, val_idx, train_df, len(train_df['gt'].unique()))
        fold_accuracies.append(acc)
    print(f"Average Validation Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")

if __name__ == '__main__':
    main()

Total valid training samples: 6828


Training:   0%|          | 0/5 [00:00<?, ?it/s]


Training Fold 1
Training fold 1 from scratch with CurricularFace


Fold 1:   3%|▎         | 1/30 [01:02<30:24, 62.92s/it, Train Acc=3.75%, Loss=23.9019, Val Acc=1.98%]

Model Saved With Accuracy1.98%


Training Epoch 2/30:  56%|█████▌    | 96/171 [00:31<00:24,  3.01it/s, loss=20.2948]